# AI Meeting Summarizer — NLP Experiments & Comparison

Welcome to the experimentation notebook for the **AI Meeting Summarizer** project!

### Project Objective
Compare two fundamental NLP summarization paradigms on realistic meeting transcripts:
1. **Extractive Summarization (TF-IDF)**: Selects the most important verbatim sentences using term-frequency statistics.
2. **Abstractive Summarization (DistilBART)**: Rewrites and synthesizes the discussion in its own words using a deep sequence-to-sequence transformer model.

## 1. Setup & Environment Verification

In [ ]:
import os
import sys

# Ensure project root is in sys.path so we can import from src
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

## 2. Load Sample Meeting Transcript
Let's load our benchmark product team standup transcript from `data/transcripts/sample_meeting.txt`.

In [ ]:
transcript_path = os.path.join(project_root, "data", "transcripts", "sample_meeting.txt")

with open(transcript_path, "r", encoding="utf-8") as f:
    raw_transcript = f.read()

words = raw_transcript.split()
print(f"Loaded transcript: {len(raw_transcript)} characters, {len(words)} words.\n")
print("--- TRANSCRIPT PREVIEW (First 400 chars) ---")
print(raw_transcript[:400] + "...")

## 3. Pipeline A: TF-IDF Extractive Summarization

### Step-by-Step Mechanism:
1. **Sentence Splitting**: Break the raw text into individual sentences.
2. **NLP Preprocessing**: Lowercase, remove punctuation, strip stopwords, and lemmatize tokens.
3. **TF-IDF Computation**: Compute the Term Frequency-Inverse Document Frequency matrix using `scikit-learn`.
4. **Sentence Scoring**: Sum the TF-IDF weights for each sentence to determine informational salience.
5. **Selection & Reordering**: Pick the top $N$ scoring sentences and sort them back into their original chronological order so the summary flows naturally.

> **Crucial Insight**: Scoring is calculated on the cleaned text, but sentences are extracted from the **original transcript** to ensure natural grammar.

In [ ]:
from src.preprocess_tfidf import preprocess_transcript
from src.tfidf_summarizer import generate_tfidf_summary

# 1. Demonstrate Preprocessing
original_sentences, cleaned_sentences = preprocess_transcript(raw_transcript)
print(f"Total sentences identified: {len(original_sentences)}\n")
print(f"Original sentence 5:   {original_sentences[4]}")
print(f"Preprocessed sentence: {cleaned_sentences[4]}\n")

# 2. Generate TF-IDF Summary
tfidf_result = generate_tfidf_summary(raw_transcript, num_sentences=4, num_key_points=4)

print("=" * 60)
print("EXTRACTIVE SUMMARY (TF-IDF):")
print("=" * 60)
print(tfidf_result["summary"])

print("\n--- EXTRACTED KEY POINTS ---")
for i, pt in enumerate(tfidf_result["key_points"], 1):
    print(f"{i}. {pt}")

## 4. Pipeline B: DistilBART Abstractive Summarization

### Step-by-Step Mechanism:
1. **Minimal Text Normalization**: Whitespace collapse and punctuation cleanup (no stopword removal or lemmatization, as transformer attention mechanisms rely on syntax and grammar).
2. **Tokenization**: Encode text with BART's BPE (Byte-Pair Encoding) tokenizer.
3. **Model Generation**: Pass tokens through the `sshleifer/distilbart-cnn-12-6` encoder-decoder architecture with beam search (`num_beams=4`).
4. **Decoding**: Convert output token IDs back into human-readable text.

In [ ]:
from src.bart_summarizer import load_bart_model, generate_bart_summary

print("Loading DistilBART model & tokenizer...")
bart_model, bart_tokenizer = load_bart_model()

# Generate Abstractive Summary
bart_result = generate_bart_summary(raw_transcript, model=bart_model, tokenizer=bart_tokenizer)

print("=" * 60)
print("ABSTRACTIVE SUMMARY (DistilBART):")
print("=" * 60)
print(bart_result["summary"])

print("\n--- EXTRACTED KEY POINTS ---")
for i, pt in enumerate(bart_result["key_points"], 1):
    print(f"{i}. {pt}")

## 5. Comparative Analysis: TF-IDF vs BART

| Dimension | TF-IDF (Extractive) | DistilBART (Abstractive) |
|---|---|---|
| **Core Concept** | Statistical word frequency weighting | Neural seq2seq transformer language model |
| **Output Nature** | Verbatim sentences from original meeting | Newly synthesized, paraphrased sentences |
| **Hallucination Risk** | **0% (Zero)** — never invents facts | **Low to Moderate** — can misattribute facts if text is ambiguous |
| **Inference Speed** | **< 10 milliseconds** | **~5-15 seconds** (CPU) |
| **Memory Footprint** | Negligible (< 10 MB) | ~600 MB (DistilBART) to ~1.6 GB (BART Large) |
| **Hardware Requirement** | Any basic CPU | Moderate CPU or optional GPU |
| **Best Use Case** | Legal, medical, compliance transcripts where wording must remain exact | Executive briefs, quick digests where concise synthesis is preferred |